> **Notebook-first lesson.** Run cells top-to-bottom. Environment-specific operations are written to be inspectable even when a service/device is unavailable.

# Lesson 61: GPU performance engineering

## Goal
Learn why some training jobs underuse expensive hardware.

## Bottlenecks
- small batch sizes
- slow data loading
- CPU preprocessing
- host-device transfer
- synchronization
- memory pressure
- inefficient tensor shapes
- excessive Python overhead

## Tools/concepts
- pinned memory
- DataLoader workers
- mixed precision
- gradient accumulation
- profiling
- memory measurement

## Mixed precision


In [ ]:
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(xb)
    loss = loss_fn(logits, yb)



Use the appropriate scaler/workflow for your PyTorch version and hardware.

## Exercise
Profile a training loop before and after changing batch size, DataLoader workers and mixed precision.

## Rule
Optimize measured bottlenecks, not imagined ones.


## Runnable activity
Run this experiment and change at least one data, threshold, deployment, or systems assumption.

In [ ]:
import time, torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device",device)
for batch in [32,256,2048]:
    x=torch.randn(batch,1024,device=device)
    W=torch.randn(1024,512,device=device)
    if device.type=="cuda": torch.cuda.synchronize()
    t=time.perf_counter()
    for _ in range(20): y=x@W
    if device.type=="cuda": torch.cuda.synchronize()
    print("batch",batch,"20 matmuls sec",time.perf_counter()-t)
print("Measure before optimizing. Small workloads may not benefit from GPU transfer/launch overhead.")

## Engineering checkpoint
Record the metric/result, the assumption you changed, and what would make this experiment invalid in a real deployment.